In [ ]:
import sys
sys.path.append('/home/projects/nyosef/zvise/PixelGen/')

In [ ]:
from PixelGen.multimodalvi import MultiModalSCVI
from PixelGen.multimodalvae import MultiModalVAE, AggMethod, D
from PixelGen.enums import AggMethod, D
from PixelGen.metrics import MultiModalVIMetrics

import anndata as ad
import pixelator
import torch
import scvi
import scipy
# from scvi import autotune

import seaborn as sns
import scanpy as sc
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
from tqdm import tqdm

from scib_metrics.benchmark import Benchmarker, BioConservation, BatchCorrection

# import ray
# from ray import tune


from PixelGen.pxl_utils import train_model, get_model_latents
from PixelGen.scvi_utils import plot_losses, pca_neighbors_umap, calc_PCA
from pixelator.common.statistics import clr_transformation, dsb_normalize


from pixelator.pna.plot import molecule_rank_plot
# from pixelator.plot import molecule_rank_plot, cell_count_plot, scatter_umi_per_upia_vs_tau
# from pixelator.statistics import clr_transformation
# from pixelator.analysis.normalization import dsb_normalize


from sklearn.preprocessing import StandardScaler, MinMaxScaler 

from PixelGen.pxl_utils import train_model, get_model_latents, convert_polarization_to_feature_matrix, \
     convert_colocalization_to_feature_matrix, download_pxl
from PixelGen.scvi_utils import plot_losses, pca_neighbors_umap, calc_PCA, add_one_hot_encoding_obsm, plot_cumulative_variance
from PixelGen.common_utils import standardize, std_clip, filter_hv, split_pair_column, filter_df_by_two_columns, rank_plot
from PixelGen.metrics import MultiModalVIMetrics, distr_autocorrelation_in_latent
from PixelGen.multimodalvi import MultiModalSCVI
from PixelGen.multimodalvae import MultiModalVAE, AggMethod, D
from PixelGen.enums import AggMethod, D

import tempfile

from scvi import REGISTRY_KEYS
from scvi.module.base import (
    BaseModuleClass,
    LossOutput,
    PyroBaseModuleClass,
    auto_move_data,
)
from torch.distributions import NegativeBinomial, Normal, Poisson, MixtureSameFamily, Beta
from torch.distributions import kl_divergence as kl



# from cytovi import CytoVI

print(torch.cuda.is_available())


scvi.settings.seed = 0
print("Last run with scvi-tools version:", scvi.__version__)
sc.set_figure_params(figsize=(6, 6), frameon=False)
sns.set_theme()
torch.set_float32_matmul_precision("high")
save_dir = tempfile.TemporaryDirectory()

%config InlineBackend.print_figure_kwargs={"facecolor": "w"}
%config InlineBackend.figure_format="retina"
%load_ext autoreload
%autoreload 2

## DATA LOADING

In [ ]:
adata=ad.read_h5ad('/home/projects/nyosef/zvise/PixelGen/PixelGen/adata_final.h5ad')
adata = adata[~adata.obs['sample'].str.contains('5to1')].copy()
adata.layers["clr"] = clr_transformation(
    adata.to_df(), axis=1, non_negative=False
)
adata

## ONLY ABUNDANCE

In [ ]:
model_cls = MultiModalSCVI

latent_name = 'abundance_model'
ab_layer='clr'

setup_kwargs = dict(layer=ab_layer, n_modalities=1, batch_key=None, )
model_kwargs = dict(n_latent=30, n_hidden=128, n_layers=1, dropout_rate=0.1, 
                        distrs=[D.Normal,], 
                        
                        loss_weights='auto',
                        
                        external_kl_weight=1,
                        decoder_kwargs=dict(decoder_param_eps=1e-2, decoder_activation='exp')
                    )
train_kwargs = dict(train_size=0.8, check_val_every_n_epoch=1, early_stopping=True, 
                    early_stopping_patience=200, batch_size=2000,
                    max_epochs=10000, enable_checkpointing=True, 
                    plan_kwargs=dict(lr=1e-4, optimizer='Adam', n_epochs_kl_warmup=400)
                )
abundance_model = train_model(adata, model_cls=model_cls, setup_kwargs=setup_kwargs, model_kwargs=model_kwargs, train_kwargs=train_kwargs,)

modalities_latent_names=[('joint', latent_name)]
get_model_latents(adata, abundance_model, modalities_latent_names=modalities_latent_names)

for key, name in modalities_latent_names:
    title = name
    fig = pca_neighbors_umap(adata, name, umap_pl_kwargs=dict(color=['sample', 'cell_type'], vcenter=0, cmap='RdBu_r', layer=ab_layer)).suptitle(title)

In [ ]:
for key, name in modalities_latent_names:
    title = name
    fig = pca_neighbors_umap(adata, name, umap_pl_kwargs=dict(color=['condition', 'cell_group'], vcenter=0, cmap='RdBu_r', layer=ab_layer)).suptitle(title)

## ONLY SPATIAL

In [ ]:
adata.obsm['spatial'].shape

In [ ]:
protein_matrix = adata.obsm['spatial']
protein_names = protein_matrix.columns if hasattr(protein_matrix, 'columns') else [f"protein_{i}" for i in range(protein_matrix.shape[1])]

# Create temporary AnnData for HVG selection
adata_tmp = sc.AnnData(
    X=protein_matrix.values if hasattr(protein_matrix, 'values') else protein_matrix,
    obs=adata.obs.copy(),
    var=pd.DataFrame(index=protein_names)
)

# Run HVG selection
sc.pp.highly_variable_genes(
    adata_tmp,
    flavor="seurat",
    n_top_genes=159
)

# Subset to HV proteins
hv_mask = adata_tmp.var["highly_variable"]
adata_hvg = adata_tmp[:, hv_mask].copy()

# Store reduced data in the original adata layers
adata.layers["spatial_hvg"] = adata_hvg.X

print("Stored layer 'spatial_hvg' with shape:", adata.layers["spatial_hvg"].shape)

In [ ]:
from sklearn.decomposition import PCA

# Get the protein matrix
protein_matrix = adata.obsm['spatial']
X = protein_matrix.values if hasattr(protein_matrix, 'values') else protein_matrix

# Run PCA
pca = PCA(n_components=159)
X_pca = pca.fit_transform(X)  # shape: (n_cells, 159)

# Store PCA result in a new layer
adata.layers["spatial_pca"] = X_pca

print("Stored layer 'spatial_pca_159' with shape:", adata.layers["spatial_pca"].shape)


In [ ]:
adata.layers

In [ ]:
model_cls = MultiModalSCVI

latent_name = 'spatial_hvg_model'
ab_layer='dsb'
spatial_key='spatial_hvg'

setup_kwargs = dict(layer=spatial_key, n_modalities=1, batch_key=None, )
model_kwargs = dict(n_latent=30, n_hidden=128, n_layers=1, dropout_rate=0.1, 
                        distrs=[D.Normal,], 
                        
                        loss_weights='auto',
                        
                        external_kl_weight=1,
                        decoder_kwargs=dict(decoder_param_eps=1e-2, decoder_activation='exp')
                    )
train_kwargs = dict(train_size=0.8, check_val_every_n_epoch=1, early_stopping=True, 
                    early_stopping_patience=200, batch_size=2000,
                    max_epochs=10000, enable_checkpointing=True, 
                    plan_kwargs=dict(lr=1e-4, optimizer='Adam', n_epochs_kl_warmup=400)
                )
spatial_hvg_model = train_model(adata, model_cls=model_cls, setup_kwargs=setup_kwargs, model_kwargs=model_kwargs, train_kwargs=train_kwargs,)

modalities_latent_names=[('joint', latent_name)]
get_model_latents(adata, spatial_hvg_model, modalities_latent_names=modalities_latent_names)


In [ ]:
for key, name in modalities_latent_names:
    title = name
    fig = pca_neighbors_umap(adata, name, umap_pl_kwargs=dict(color=['condition', 'cell_group','cell_type'], vcenter=0, cmap='RdBu_r', layer=ab_layer)).suptitle(title)

In [ ]:
model_cls = MultiModalSCVI

latent_name = 'spatial_pca_model'
ab_layer='dsb'
spatial_key='spatial_pca'

setup_kwargs = dict(layer=spatial_key, n_modalities=1, batch_key=None, )
model_kwargs = dict(n_latent=30, n_hidden=128, n_layers=1, dropout_rate=0.1, 
                        distrs=[D.Normal,], 
                        
                        loss_weights='auto',
                        
                        external_kl_weight=1,
                        decoder_kwargs=dict(decoder_param_eps=1e-2, decoder_activation='exp')
                    )
train_kwargs = dict(train_size=0.8, check_val_every_n_epoch=1, early_stopping=True, 
                    early_stopping_patience=200, batch_size=2000,
                    max_epochs=10000, enable_checkpointing=True, 
                    plan_kwargs=dict(lr=1e-4, optimizer='Adam', n_epochs_kl_warmup=400)
                )
spatial_pca_model = train_model(adata, model_cls=model_cls, setup_kwargs=setup_kwargs, model_kwargs=model_kwargs, train_kwargs=train_kwargs,)

modalities_latent_names=[('joint', latent_name)]
get_model_latents(adata, spatial_pca_model, modalities_latent_names=modalities_latent_names)



In [ ]:
for key, name in modalities_latent_names:
    title = name
    fig = pca_neighbors_umap(adata, name, umap_pl_kwargs=dict(color=['condition', 'cell_group'], vcenter=0, cmap='RdBu_r', layer=ab_layer)).suptitle(title)

## shared encoder

In [ ]:
df = pd.DataFrame(
    adata.obsm['spatial_hvg'],
    index=adata.obs_names  # ensures row indices match adata
)

adata.obsm['spatial_hvg'] = df
adata.obsm['dsb'] = adata.layers['dsb']

adata.obsm

In [ ]:
model_cls = MultiModalSCVI

latent_name = 'shared_enc_model'
ab_layer='dsb'
spatial_key='spatial_hvg'

setup_kwargs = dict(layer='dsb', extra_modality_keys=['spatial_hvg'], n_modalities=2, batch_key=None, )
model_kwargs = dict(n_latent=30, n_hidden=128, n_layers=2, dropout_rate=0.1, 
                        distrs=[D.Normal, D.Normal], 
                        agg_method=AggMethod.SHARED_ENCODER,
                        loss_weights='auto',
                        joint_kl=True, unimodal_kl=False,
                        external_kl_weight=1,
                        decoder_kwargs=dict(decoder_param_eps=1e-2, decoder_activation='exp')
                    )
train_kwargs = dict(train_size=0.8, check_val_every_n_epoch=1, early_stopping=True, 
                    early_stopping_patience=200, batch_size=2000,
                    max_epochs=10000, enable_checkpointing=True, 
                    plan_kwargs=dict(lr=1e-4, optimizer='Adam', n_epochs_kl_warmup=400)
                )
shared_enc_model = train_model(adata, model_cls=model_cls, setup_kwargs=setup_kwargs, model_kwargs=model_kwargs, train_kwargs=train_kwargs,)

modalities_latent_names=[('joint', latent_name)]
get_model_latents(adata, shared_enc_model, modalities_latent_names=modalities_latent_names)


In [ ]:
for key, name in modalities_latent_names:
    title = name
    fig = pca_neighbors_umap(adata, name, umap_pl_kwargs=dict(color=['condition', 'cell_group'], 
                                                              vcenter=0, cmap='RdBu_r', layer=ab_layer)).suptitle(title)

global weights

In [ ]:
model_cls = MultiModalSCVI

latent_name = 'global_weights'
ab_layer='dsb'
spatial_key='spatial_hvg'

setup_kwargs = dict(layer=ab_layer, extra_modality_keys=[spatial_key], n_modalities=2, batch_key=None, )
model_kwargs = dict(n_latent=30, n_hidden=128, n_layers=2, dropout_rate=0.1, 
                        distrs=[D.Normal, D.Normal], 
                        
                        loss_weights='auto',
                        joint_kl=False, unimodal_kl=True,
                        
                        decoder_kwargs=dict(decoder_param_eps=1e-2, decoder_activation='exp')
                    )
train_kwargs = dict(train_size=0.8, check_val_every_n_epoch=1, early_stopping=True, 
                    early_stopping_patience=200, batch_size=2000,
                    max_epochs=10000, enable_checkpointing=True, 
                    plan_kwargs=dict(lr=1e-4, optimizer='Adam', n_epochs_kl_warmup=400)
                )
global_w_model = train_model(adata, model_cls=model_cls, setup_kwargs=setup_kwargs, model_kwargs=model_kwargs, train_kwargs=train_kwargs,)



In [ ]:
modalities_latent_names=[(s,f'{s}_latent') for s in ('joint', 'dsb', setup_kwargs['extra_modality_keys'][0])]
get_model_latents(adata, global_w_model, modalities_latent_names=modalities_latent_names)

In [ ]:
for key, name in modalities_latent_names:
    title = name
    fig = pca_neighbors_umap(adata, name, umap_pl_kwargs=dict(color=['condition',  'cell_group'], cmap='RdBu_r')).suptitle(title)

## METRICS

In [ ]:
adata.obsm

In [ ]:
add_one_hot_encoding_obsm(adata, obs_column='cell_type')

metrics = MultiModalVIMetrics(
    adata,
    models = {
        'abundance_only': abundance_model,
        'spatial_hvg': spatial_hvg_model,
        'spatial_pca': spatial_pca_model,
        
        'shared_encoder': shared_enc_model,
        'global_weights': global_w_model
    },
    pca_key='shared_enc_model_pca',
    additional_autocorr_keys=['cell_type'],
)
metrics.run()

In [ ]:
_ = metrics.mean_modality_errors_barplot()
_ = metrics.mean_modality_errors_barplot(reconstruction_mean=True)

In [ ]:
_ = metrics.mean_autocorr_barplot()
# _ = metrics.autocorr_barplot(autocorr_key=pol_key, auto_filter_features=10)
_ = metrics.autocorr_barplot(autocorr_key='spatial', features=coloc_hvg_vars[:10])
_ = metrics.autocorr_barplot(autocorr_key='cell_type', auto_filter_features=10, figsize=(10, 8))

## SEPARATED ANALYSIS

In [ ]:
adata=ad.read_h5ad('/home/projects/nyosef/zvise/PixelGen/PixelGen/adata_separated_final.h5ad')
complete_adata=ad.read_h5ad('/home/projects/nyosef/zvise/PixelGen/PixelGen/adata_final.h5ad')


adata.obs=adata.obs.merge(complete_adata.obs['sample'], right_index=True, left_on='component',how='left')
adata.obs['sample'] = adata.obs['sample_x'].combine_first(adata.obs['sample_y'])
adata.obs.drop(['sample_x', 'sample_y'], axis=1, inplace=True)


adata = adata[~adata.obs['sample'].str.contains('5to1')].copy()

adata.layers["clr"] = clr_transformation(
    adata.to_df(), axis=1, non_negative=False
)
adata


In [ ]:
adata.obs['cell_type'].value_counts()

In [ ]:
rel_types=['B_doublet','B','CD8_doublet','CD4_doublet','CD8-CAR-nonactive','CD4-CAR-nonactive']
adata=adata[adata.obs.cell_type.isin(rel_types)].copy()
adata.obs['cell_type'].value_counts()

In [ ]:
model_cls = MultiModalSCVI

latent_name = 'abundance_model'
ab_layer='clr'

setup_kwargs = dict(layer=ab_layer, n_modalities=1, batch_key=None, )
model_kwargs = dict(n_latent=30, n_hidden=128, n_layers=1, dropout_rate=0.1, 
                        distrs=[D.Normal,], 
                        
                        loss_weights='auto',
                        
                        external_kl_weight=1,
                        decoder_kwargs=dict(decoder_param_eps=1e-2, decoder_activation='exp')
                    )
train_kwargs = dict(train_size=0.8, check_val_every_n_epoch=1, early_stopping=True, 
                    early_stopping_patience=200, batch_size=2000,
                    max_epochs=10000, enable_checkpointing=True, 
                    plan_kwargs=dict(lr=1e-4, optimizer='Adam', n_epochs_kl_warmup=400)
                )
shared_enc_model = train_model(adata, model_cls=model_cls, setup_kwargs=setup_kwargs, model_kwargs=model_kwargs, train_kwargs=train_kwargs,)

modalities_latent_names=[('joint', latent_name)]
get_model_latents(adata, shared_enc_model, modalities_latent_names=modalities_latent_names)



In [ ]:
for key, name in modalities_latent_names:
    title = name
    fig = pca_neighbors_umap(adata, name, umap_pl_kwargs=dict(color=['condition', 'cell_type'], vcenter=0, cmap='RdBu_r', layer=ab_layer)).suptitle(title)

## ONLY SPATIAL

In [ ]:
adata.obsm['spatial'].shape

In [ ]:
adata.obsm['spatial'][:5, :5]  # Display the first 5 rows and columns of the spatial data  

In [ ]:
protein_matrix = adata.obsm['spatial']
protein_names = protein_matrix.columns if hasattr(protein_matrix, 'columns') else [f"protein_{i}" for i in range(protein_matrix.shape[1])]

adata_tmp = sc.AnnData(
    X=protein_matrix.values if hasattr(protein_matrix, 'values') else protein_matrix,
    obs=adata.obs.copy(),
    var=pd.DataFrame(index=protein_names)
)

sc.pp.highly_variable_genes(
    adata_tmp,
    flavor="seurat",
    n_top_genes=159
)

hv_mask = adata_tmp.var["highly_variable"]
adata_hvg = adata_tmp[:, hv_mask].copy()

adata.layers["spatial_hvg"] = adata_hvg.X

print("Stored layer 'spatial_hvg' with shape:", adata.layers["spatial_hvg"].shape)

In [ ]:
from sklearn.decomposition import PCA
protein_matrix = adata.obsm['spatial']
X = protein_matrix.values if hasattr(protein_matrix, 'values') else protein_matrix

pca = PCA(n_components=159)
X_pca = pca.fit_transform(X)  

adata.layers["spatial_pca"] = X_pca

print("Stored layer 'spatial_pca_159' with shape:", adata.layers["spatial_pca"].shape)

In [ ]:
adata.layers

In [ ]:
model_cls = MultiModalSCVI

latent_name = 'spatial_hvg_model'
ab_layer='dsb'
spatial_key='spatial_hvg'

setup_kwargs = dict(layer=spatial_key, n_modalities=1, batch_key=None, )
model_kwargs = dict(n_latent=30, n_hidden=128, n_layers=1, dropout_rate=0.1, 
                        distrs=[D.Normal,], 
                        
                        loss_weights='auto',
                        
                        external_kl_weight=1,
                        decoder_kwargs=dict(decoder_param_eps=1e-2, decoder_activation='exp')
                    )
train_kwargs = dict(train_size=0.8, check_val_every_n_epoch=1, early_stopping=True, 
                    early_stopping_patience=200, batch_size=2000,
                    max_epochs=10000, enable_checkpointing=True, 
                    plan_kwargs=dict(lr=1e-4, optimizer='Adam', n_epochs_kl_warmup=400)
                )
shared_enc_model = train_model(adata, model_cls=model_cls, setup_kwargs=setup_kwargs, model_kwargs=model_kwargs, train_kwargs=train_kwargs,)

modalities_latent_names=[('joint', latent_name)]
get_model_latents(adata, shared_enc_model, modalities_latent_names=modalities_latent_names)


In [ ]:
for key, name in modalities_latent_names:
    title = name
    fig = pca_neighbors_umap(adata, name, umap_pl_kwargs=dict(color=['condition', 'cell_type'], vcenter=0, cmap='RdBu_r', layer=ab_layer)).suptitle(title)

In [ ]:
model_cls = MultiModalSCVI

latent_name = 'spatial_pca_model'
spatial_key='spatial_pca'

setup_kwargs = dict(layer=spatial_key, n_modalities=1, batch_key=None, )
model_kwargs = dict(n_latent=30, n_hidden=128, n_layers=1, dropout_rate=0.1, 
                        distrs=[D.Normal,], 
                        
                        loss_weights='auto',
                        
                        external_kl_weight=1,
                        decoder_kwargs=dict(decoder_param_eps=1e-2, decoder_activation='exp')
                    )
train_kwargs = dict(train_size=0.8, check_val_every_n_epoch=1, early_stopping=True, 
                    early_stopping_patience=200, batch_size=2000,
                    max_epochs=10000, enable_checkpointing=True, 
                    plan_kwargs=dict(lr=1e-4, optimizer='Adam', n_epochs_kl_warmup=400)
                )
shared_enc_model = train_model(adata, model_cls=model_cls, setup_kwargs=setup_kwargs, model_kwargs=model_kwargs, train_kwargs=train_kwargs,)

modalities_latent_names=[('joint', latent_name)]
get_model_latents(adata, shared_enc_model, modalities_latent_names=modalities_latent_names)



In [ ]:
for key, name in modalities_latent_names:
    title = name
    fig = pca_neighbors_umap(adata, name, umap_pl_kwargs=dict(color=['condition', 'cell_type'], vcenter=0, cmap='RdBu_r', layer=ab_layer)).suptitle(title)

## SHARED ENCODER

In [ ]:
adata

In [ ]:
df = pd.DataFrame(
    adata.layers['spatial_hvg'],
    index=adata.obs_names  # ensures row indices match adata
)

adata.obsm['spatial_hvg'] = df

dfpca = pd.DataFrame(
    adata.layers['spatial_pca'],
    index=adata.obs_names  # ensures row indices match adata
)

adata.obsm['spatial_pca'] = dfpca

adata.obsm

In [ ]:
model_cls = MultiModalSCVI

latent_name = 'shared_enc_model'
ab_layer='clr'
spatial_key='spatial_pca'

setup_kwargs = dict(layer=ab_layer, extra_modality_keys=[spatial_key], n_modalities=2, batch_key=None, )
model_kwargs = dict(n_latent=30, n_hidden=128, n_layers=2, dropout_rate=0.1, 
                        distrs=[D.Normal, D.Normal], 
                        agg_method=AggMethod.SHARED_ENCODER,
                        loss_weights='auto',
                        joint_kl=True, unimodal_kl=False,
                        external_kl_weight=1,
                        decoder_kwargs=dict(decoder_param_eps=1e-2, decoder_activation='exp')
                    )
train_kwargs = dict(train_size=0.8, check_val_every_n_epoch=1, early_stopping=True, 
                    early_stopping_patience=200, batch_size=2000,
                    max_epochs=10000, enable_checkpointing=True, 
                    plan_kwargs=dict(lr=1e-4, optimizer='Adam', n_epochs_kl_warmup=400)
                )
shared_enc_model = train_model(adata, model_cls=model_cls, setup_kwargs=setup_kwargs, model_kwargs=model_kwargs, train_kwargs=train_kwargs,)

modalities_latent_names=[('joint', latent_name)]
get_model_latents(adata, shared_enc_model, modalities_latent_names=modalities_latent_names)


In [ ]:
for key, name in modalities_latent_names:
    title = name
    fig = pca_neighbors_umap(adata, name, umap_pl_kwargs=dict(color=['condition', 'cell_type'], 
                                                              vcenter=0, cmap='RdBu_r', layer=ab_layer)).suptitle(title)

## GLOBAL WEIGHTS

In [ ]:
model_cls = MultiModalSCVI

latent_name = 'global_weights'
ab_layer='clr'
spatial_key='spatial_pca'

setup_kwargs = dict(layer=ab_layer, extra_modality_keys=[spatial_key], n_modalities=2, batch_key=None, )
model_kwargs = dict(n_latent=30, n_hidden=128, n_layers=2, dropout_rate=0.1, 
                        distrs=[D.Normal, D.Normal], 
                        
                        loss_weights='auto',
                        joint_kl=False, unimodal_kl=True,
                        
                        decoder_kwargs=dict(decoder_param_eps=1e-2, decoder_activation='exp')
                    )
train_kwargs = dict(train_size=0.8, check_val_every_n_epoch=1, early_stopping=True, 
                    early_stopping_patience=200, batch_size=2000,
                    max_epochs=10000, enable_checkpointing=True, 
                    plan_kwargs=dict(lr=1e-4, optimizer='Adam', n_epochs_kl_warmup=400)
                )
shared_enc_model = train_model(adata, model_cls=model_cls, setup_kwargs=setup_kwargs, model_kwargs=model_kwargs, train_kwargs=train_kwargs,)



In [ ]:
modalities_latent_names=[(s,f'{s}_latent') for s in ('joint', 'clr', setup_kwargs['extra_modality_keys'][0])]
get_model_latents(adata, shared_enc_model, modalities_latent_names=modalities_latent_names)

In [ ]:
for key, name in modalities_latent_names:
    title = name
    fig = pca_neighbors_umap(adata, name, umap_pl_kwargs=dict(color=['condition',  'cell_type'], cmap='RdBu_r')).suptitle(title)